# Báo cáo Tuần 1 (W1-02) – CameraTrapML\n**Người thực hiện:** TV2 (Phụ trách Baseline & Evaluation)\n**Mục tiêu:** Kiểm tra môi trường (PyTorch, CUDA), và định nghĩa khung đánh giá (Evaluation Framework) chuẩn cho dự án.

In [1]:
import sys
import platform
import datetime
import torch

print("=" * 72)
print("1. THÔNG TIN PYTHON & HỆ ĐIỀU HÀNH")
print("=" * 72)
print(f"  Python version  : {sys.version}")
print(f"  Platform        : {platform.platform()}")

print("\n" + "=" * 72)
print("2. PYTORCH & CUDA")
print("=" * 72)
print(f"  PyTorch version : {torch.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA version    : {torch.version.cuda or 'N/A'}")

print("\n" + "=" * 72)
print("3. THÔNG TIN GPU")
print("=" * 72)
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"  Số GPU phát hiện: {num_gpus}")
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        vram_total_gb = props.total_mem / (1024 ** 3)
        print(f"\n  --- GPU {i} ---")
        print(f"  Tên             : {props.name}")
        print(f"  VRAM tổng       : {vram_total_gb:.2f} GB")
else:
    print("  [CẢNH BÁO] Không phát hiện GPU CUDA nào. Sẽ chạy trên CPU.")

print("\n

In [2]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix as sk_confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

def compute_metrics(y_true, y_pred, class_names):
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    report_str = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    return {
        "macro_f1": macro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "accuracy": accuracy,
        "classification_report_str": report_str,
    }

def plot_confusion_matrix(y_true, y_pred, class_names):
    cm = sk_confusion_matrix(y_true, y_pred)
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    cm_display = cm.astype(float) / row_sums

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cm_display, annot=True, fmt=".2%", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
        linewidths=0.5, linecolor="gray", ax=ax
    )
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_title("Confusion Matrix\n

### Kết luận & Giao thức đánh giá\n- **Môi trường:** GPU đã sẵn sàng cho huấn luyện mô hình.\n- **Chỉ số (Metric):** `Macro-F1` được sử dụng làm tiêu chí chính để chọn mô hình (Checkpoint Metric) và Early Stopping do tính mất cân bằng lớp nghiêm trọng (như Heo rừng chiếm đa số, Mang lớn/Thỏ vằn là thiểu số).\n- **Cô lập Test Set:** Không tune tham số, phân tích trên tập Test. Chỉ mở vào Tuần 4.\n- **So sánh B1 vs B2:** Cần sử dụng chung một tập Test (chia theo site-level) và cùng random seed = 42.